<a href="https://colab.research.google.com/github/Meghana-506/NASSCOM/blob/main/WEEK%203/U8_Data_Cleaning_Part1_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Core imports for the whole lab
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
print('Setup complete. pandas', pd.__version__)

Setup complete. pandas 2.2.3


In [10]:
raw = pd.DataFrame({
    'id':    [1, 2, 3, 4, 5, 6, 7, 7],
    'name':  ['Ana', 'Bo', 'Cy', 'Di', 'Eve', 'Fin', 'Gus', 'Gus'],
    'age':   [30, 25, np.nan, 41, -1, 38, 29, 29],
    'city':  [' Pune ', 'pune', 'DELHI', 'Delhi ', 'Mumbai', 'bombay', 'Pune.', 'Pune.'],
    'spend': ['120.5', '80.0', '200.2', 'N/A', '150.0', '99000', '110.0', '110.0'],
    'date':  ['2024-01-05', '2024-01-06', '2024-01-07', '2024-01-08',
              '2024-01-09', '2024-01-10', '2024-01-11', '2024-01-11'],
})
raw

,id,name,age,city,spend,date
0,1,Ana,30.0,Pune,120.5,2024-01-05
1,2,Bo,25.0,pune,80.0,2024-01-06
2,3,Cy,NaN,DELHI,200.2,2024-01-07
3,4,Di,41.0,Delhi,N/A,2024-01-08
4,5,Eve,-1.0,Mumbai,150.0,2024-01-09
5,6,Fin,38.0,bombay,99000,2024-01-10
6,7,Gus,29.0,Pune.,110.0,2024-01-11
7,7,Gus,29.0,Pune.,110.0,2024-01-11


In [11]:
df = raw.copy()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      8 non-null      int64  
 1   name    8 non-null      object 
 2   age     7 non-null      float64
 3   city    8 non-null      object 
 4   spend   8 non-null      object 
 5   date    8 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes


In [12]:
print('Missing per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nNote: spend is type', df['spend'].dtype, "-> stored as text!")


Missing per column:
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64

Duplicate rows: 1

Note: spend is type object -> stored as text!


In [13]:
#Lab Exercise 1
#1.duplicate row count
print(df.duplicated().sum())
#2.for missing count per column
print(df.isna().sum())

1
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64


In [14]:
# 'N/A' (in spend) and -1 (in age) are really missing -> make them NaN
df['spend'] = pd.to_numeric(df['spend'], errors='coerce')  # 'N/A' -> NaN, text -> number
df['age']   = df['age'].replace(-1, np.nan)                # sentinel -> NaN

print('Missing after unmasking:')
print(df[['age', 'spend']].isna().sum())


Missing after unmasking:
age      2
spend    1
dtype: int64


In [15]:
# median is robust to skew/outliers -> good for 'spend' and 'age'
df['age']   = df['age'].fillna(df['age'].median())
df['spend'] = df['spend'].fillna(df['spend'].median())
print('Missing after imputing:', df[['age', 'spend']].isna().sum().sum())

Missing after imputing: 0


In [16]:
ex = raw.copy()

# 1. Unmask missing values
ex['spend'] = pd.to_numeric(ex['spend'], errors='coerce')
ex['age'] = ex['age'].replace(-1, np.nan)


# 2a. dropna version
drop_version = ex.dropna()


# 2b. median-impute version
impute_version = ex.copy()

# Fill numerical missing values with their median
impute_version['spend'] = impute_version['spend'].fillna(
    impute_version['spend'].median()
)

impute_version['age'] = impute_version['age'].fillna(
    impute_version['age'].median()
)


# 3. Compare row counts
print("Original rows:", len(ex))
print("Rows after dropna:", len(drop_version))
print("Rows after median imputation:", len(impute_version))

print("Rows lost by dropping:", len(ex) - len(drop_version))

Original rows: 8
Rows after dropna: 5
Rows after median imputation: 8
Rows lost by dropping: 3


In [17]:
print('Before:', df.shape)
df = df.drop_duplicates()
print('After :', df.shape, '-> removed the repeated Gus row')

Before: (8, 6)
After : (7, 6) -> removed the repeated Gus row


In [18]:
# 'date' is text -> convert to real datetimes so sorting/maths work
df['date'] = pd.to_datetime(df['date'])
# 'city' is a category -> mark it as such (saves memory, signals intent)
df['city'] = df['city'].astype('string')
print(df.dtypes)


id                int64
name             object
age             float64
city     string[python]
spend           float64
date     datetime64[ns]
dtype: object


In [19]:
ex = raw.copy()

# 1. Convert spend to numeric and date to datetime
ex['spend'] = pd.to_numeric(ex['spend'], errors='coerce')
ex['date'] = pd.to_datetime(ex['date'], errors='coerce')


# 2. Drop duplicate rows
ex = ex.drop_duplicates()


# 3. Print data types and final shape
print("Data types:")
print(ex.dtypes)

print("\nFinal shape:")
print(ex.shape)

Data types:
id                int64
name             object
age             float64
city             object
spend           float64
date     datetime64[ns]
dtype: object

Final shape:
(7, 6)


In [20]:
# spend has a 99000 value among ~100s -> a clear outlier
q1, q3 = df['spend'].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f'Q1={q1:.1f}  Q3={q3:.1f}  IQR={iqr:.1f}')
print(f'Normal range: {low:.1f} to {high:.1f}')

outliers = df[(df['spend'] < low) | (df['spend'] > high)]
print('\nOutlier rows:')
print(outliers[['name', 'spend']])

Q1=115.2  Q3=175.1  IQR=59.8
Normal range: 25.5 to 264.9

Outlier rows:
  name    spend
5  Fin  99000.0


In [21]:
# clip values to the IQR bounds instead of deleting the row
df['spend_capped'] = df['spend'].clip(lower=low, upper=high)
print(df[['name', 'spend', 'spend_capped']])

  name    spend  spend_capped
0  Ana    120.5       120.500
1   Bo     80.0        80.000
2   Cy    200.2       200.200
3   Di    120.5       120.500
4  Eve    150.0       150.000
5  Fin  99000.0       264.875
6  Gus    110.0       110.000


In [22]:
# 1. Q1, Q3, IQR for 'age'
Q1 = df['age'].quantile(0.25)
Q3 = df['age'].quantile(0.75)
IQR = Q3 - Q1

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)


# 2. Lower & upper bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)


# 3. Rows outside the bounds
outliers = df[(df['age'] < lower_bound) | (df['age'] > upper_bound)]

print("Outliers:")
print(outliers)

Q1: 29.25
Q3: 34.0
IQR: 4.75
Lower bound: 22.125
Upper bound: 41.125
Outliers:
Empty DataFrame
Columns: [id, name, age, city, spend, date, spend_capped]
Index: []


In [23]:
print(df['city'].value_counts())   # ' Pune ', 'pune', 'Pune.' all look different!

city
 Pune     1
pune      1
DELHI     1
Delhi     1
Mumbai    1
bombay    1
Pune.     1
Name: count, dtype: Int64


In [24]:
s = df['city'].astype('string')
s = s.str.strip()                       # trim whitespace
s = s.str.lower()                       # unify case
s = s.str.replace('.', '', regex=False) # drop stray punctuation
s = s.replace({'bombay': 'mumbai'})     # map known variants to one label
df['city'] = s
print(df['city'].value_counts())        # now clean categories


city
pune      3
delhi     2
mumbai    2
Name: count, dtype: Int64


In [25]:
messy = pd.Series(
    [' London ', 'london', 'LONDON', 'N.Y.', 'new york ', 'New York'],
    dtype='string'
)

# 1. Strip whitespace + lowercase
messy = messy.str.strip().str.lower()


# 2. Map 'n.y.' -> 'new york'
messy = messy.replace('n.y.', 'new york')


# 3. Value counts
print(messy.value_counts())

london      3
new york    3
Name: count, dtype: Int64
